# Julia notebook to compute the symbolic solution to the frictional geostrophic equations with a specified buoyancy field and surface wind stress.

twnh Nov '25

This notebook solves

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + \epsilon^2   \nu \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + \epsilon^2   \nu \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given viscosity $\nu = \nu_0$, and boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \epsilon^2 \nu \partial_z u & = \tau^x \\
\displaystyle  \epsilon^2 \nu \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H(x,y)) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code derives the equation satisfied by the surface pressure field $p_s(x,y)$.
It then solves for the pressure field given a specific simple example of specified windstress and simple buoyancy forcing.

This code was used for the Paris Arctic Dynamics Workshop talk in November 2025.

In [ ]:
using SymPy
im = SymPy.im  # SymPy's imaginary unit
notebook_name = "ExampleSolution_v0.4"
using Infiltrator
using Gridap
using GridapGmsh
using GridapMakie
using Gridap.Geometry
using Gridap.ReferenceFEs
using WriteVTK
using Profile
using ProfileView
# using ForwardDiff
# using Surrogates
# using Zygote
# using Random

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
z, ξ   = symbols("z ξ",   real=true, negative=true) # Vertical coordinate and source location (both in [-H,0])
x, y   = symbols("x y",   real=true)                # Horizontal coordinates
H      = SymFunction("H", real=true, positive=true) # Domain depth H(x)
geometry_params = (H(x,y), z, ξ)

# Frictional thermal wind equation symbolic parameters:
f, ϵ   = symbols("f ϵ",   real=true, positive=true) # Coriolis parameter and Ekman number
ν₀, ϕ  = symbols("ν₀ ϕ",  real=true, positive=true) # Viscosity parameters
τs     = SymFunction("τs",    complex=true)         # Complex surface wind stress
psg    = SymFunction("psg",   complex=true)         # Surface pressure gradient (∂/∂x + i ∂/∂y) pₛ(x,y)
b      = SymFunction("b",     real=true)            # Buoyancy field b(x,y,z)

# Define symbolic viscosity here:
ν = ν₀                                              # Constant viscosity profile
uv_params = (f, ϵ, ν) 

# Buoyancy equation symbolic parameters:
κ₀, ψ  = symbols("κ₀ ψ",  real=true, positive=true) # Diffusivity parameters
γ      = symbols("γ",     real=true, positive=true) # Relaxation parameter
B      = SymFunction("B", real=true)                # Relaxation buoyancy field

# Define diffusivity profile here:
κ = κ₀                                              # Constant viscosity profile

b_params = (κ, ϵ, γ) ;

### Compute pressure fields for the specified buoyancy field $b$:

In [ ]:
# Define buoyancy gradient field:
bg     = SymFunction("bg",    complex=true)         # Buoyancy gradient

# Compute pressure field:
pbarog = SymPy.integrate(bg(x,y,ξ),(ξ,z,0))         # Baroclinic presure gradient 
pg     = psg(x,y) + pbarog                          # Total pressure gradient
pbotg  = pg.subs(z,-H(x,y))                         # Bottom pressure gradient
# χ      = - SymPy.integrate(z * b(x,y,z),(z,-H(x,y),0))    # Baroclinic potential energy (diagnostic)

### Function to solve the frictional geostrophic equation using a Green's function:

In [ ]:
function compute_Guv(uv_params, geometry_params)
    # Setup symbols and parameters:
    f, ϵ, ν = uv_params
    H, z, ξ = geometry_params
    uv      = SymFunction("uv")
    A       = symbols("A",real=true)                                            # Unknown coefficient in the Green's function solution

    #0. Define the ODE for d/dz(uv(z)) = uv(z):
    ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)
    
    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    Gₘ = dsolve(ode, uv(z), ics = Dict(uv(z).subs(z,-H(x,y))=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₘ)) == 0                               # Check solution
    # Replace constant names because otherwise they can interfere with the constants from the next dsolve below.
    const_names = collect([string(s) for s in Gₘ.free_symbols if occursin(r"^C\d+", string(s))])
    Gₘ = Gₘ.subs(const_names[1],A)
    @assert simplify(Gₘ.subs(z,-H(x,y))) == 0                                   # Check bottom BC

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    Gₚ = dsolve(ode, uv(z), ics = Dict(diff(uv(z),z).subs(z,0)=>0)).rhs
    @assert simplify(ode.lhs.subs(uv(z),Gₚ)) == 0                               # Check solution
    @assert simplify(diff(Gₚ,z).subs(z,0)) == 0                                 # Check surface BC

    # 3. Compute Wronskian $W(z)$:
    W = Gₘ * diff(Gₚ, z) - Gₚ * diff(Gₘ, z)

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = Gₘ * Gₚ.subs(z,ξ) / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))
    Gp = Gₘ.subs(z,ξ) * Gₚ / (ϵ^2 * ν.subs(z,ξ) * W.subs(z,ξ))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * ν.subs(z,ξ)) == 0

    # #5. Define piecewise Green's function:
    G = sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ)))
    
    # Check boundary conditions:
    @assert simplify(diff(G,z).subs(z,0).subs(ξ,-H//2)) == 0
    @assert simplify(G.subs(z,-H(x,y)).subs(ξ,-H//2)) == 0

    # Final simplify (to cancel constants). Avoid simplify in general because it's not always reproducible.
    G = simplify(G)
    return G
end ;

### Compute the velocity G's function:

In [ ]:
Guv_sym = compute_Guv(uv_params,geometry_params) 
Guv = Guv_sym.subs(f,ϕ^2 * ϵ^2 * ν₀)
Guv0 = Guv.subs(ξ,0) 
tmp = sympy.integrate(expand(Guv), (ξ, -H(x,y), 0)).args[1].args[1]
max_obj = sympy.Max(z, -H(x, y))
Guv_int_wrt_ξ = tmp.subs(max_obj, z)

#### Function to solve the buoyancy equation using a Green's function:

In [ ]:
function compute_Gb(b_params, geometry_params)
    # Setup symbols and parameters:
    κ, ϵ, γ = b_params
    H, z, ξ = geometry_params
    b       = SymFunction("b")
    A       = symbols("A",     real=true)                      # Unknown coefficient in the Green's function solution
    
    #0. Define the ODE for b(z):
    ode = Eq( - γ * b(z)     + ϵ^2 * diff(diff(κ *  b(z),z),z), 0)
    # ode = Eq(-im * f * uv(z) + ϵ^2 * diff(diff(ν * uv(z),z),z), 0)

    # 1. Solve for $G_- (z)$ on $-H \le z \le \xi$:
    bm = dsolve(ode, b(z), ics = Dict(diff(b(z),z).subs(z,-H)=>0)).rhs
    @assert simplify(ode.lhs.subs(b(z),bm)) == 0                  # Check solution
    @assert simplify(diff(bm,z).subs(z,-H) - 0) == 0              # Check Neumann BC at bottom
    bm_const = filter(x -> startswith(string(x), "C"), bm.free_symbols)
    bm = bm.subs(first(bm_const), A)                              # Replace constant with A so it doesn't conflict later

    # 2. Solve for $G_+(z)$ on  $\xi \le z \le 0$:
    bp = dsolve(ode, b(z), ics = Dict(b(0)=>0) ).rhs
    @assert simplify(ode.lhs.subs(b(z),bp)) == 0                  # Check solution
    @assert simplify(bp.subs(z,0)) == 0                           # Check Dirichlet BC at top

    # 3. Compute Wronskian $W(z)$:
    W = simplify(bm * diff(bp, z) - bp * diff(bm, z))

    # 4. Compute Green's function $G(z; \xi)$:
    Gm = simplify(bm * bp.subs(z,ξ) / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))
    Gp = simplify(bm.subs(z,ξ) * bp / (ϵ^2 * κ.subs(z,ξ) * W.subs(z,ξ)))

    # Check continuity and jump condition:
    @assert Gm.subs(z,ξ) - Gp.subs(z,ξ) == 0
    @assert simplify(diff(Gm, z).subs(z,ξ) - diff(Gp, z).subs(z,ξ)) + 1/(ϵ^2 * κ.subs(z,ξ)) == 0

    # Define piecewise Green's function:
    G = simplify(sympy.Piecewise((Gm, Le(z,ξ)), (Gp, Ge(z,ξ))))

    # Check boundary conditions are satisfied
    @assert simplify(diff(G,z).subs(z,-H).subs(ξ,-H//3).subs(H,1//2)) == 0
    @assert simplify(G.subs(z,0).subs(ξ,-H//3)) == 0

    return G
end

### Compute the buoyancy Green's function:

In [ ]:
Gb_sym  = compute_Gb(b_params,geometry_params)
Gb = Gb_sym.subs(γ,ψ^2 * ϵ^2 * κ₀)

### Symbolic computation of buoyancy $b$

In [ ]:
# Compute buoyancy field b(z):
b_sym = SymPy.integrate(expand(-Gb * γ * B(x,y,ξ)), (ξ,-H(x,y),0) )
b = simplify(b_sym.subs(max_obj,z))

# Check that the final expression for b satisfies the original differential equation:
tmp1 = ϵ^2 * diff(diff(κ * b,z),z) - γ * b
tmp1 = simplify(tmp1.subs(ψ,sqrt(γ/κ₀)/ϵ))
@assert tmp1 + γ * B(x,y,z) == 0

### Symbolic computation of flow $(u(z),v(z)), U, V, \tau_b$

In [ ]:
# Compute flow field u(z), v(z):
𝔲1 = SymPy.integrate(expand(Guv * pg.subs(z,ξ)),(ξ,-H(x,y),0))
𝔲1 = 𝔲1.subs(max_obj,z)
𝔲2 = - Guv0 * τs(x,y)
𝔲 = 𝔲1 + 𝔲2

# Check that the final expression for 𝔲 satisfies the original differential equation:
tmp1 = -im * f * 𝔲1 + ϵ^2 * diff(diff(ν * 𝔲1,z),z)
tmp1 = simplify(tmp1.subs(ϕ,sqrt(f/ν₀)/ϵ))
tmp2 = -im * f * 𝔲2 + ϵ^2 * diff(diff(ν * 𝔲2,z),z)
tmp2 = simplify(tmp2.subs(ϕ,sqrt(f/ν₀)/ϵ))
@assert tmp1 + tmp2 == pg

# Compute depth-integrated flow:
𝔘 = SymPy.integrate(expand(𝔲),(z,-H(x,y),0))

# Compute bottom stress on fluid:
τb = - ϵ^2 * ν * diff(𝔲,z).subs(z,-H(x,y))

# Check expression for surface stress on fluid:
@assert simplify(diff(𝔲1,z).subs(z,0)) == 0     # Pressure-driven part of surface stress vanishes
tmp = ϵ^2 * ν * diff(𝔲,z).subs(z,0)
@assert simplify(tmp - τs(x,y)) == 0

### Check final results from LaTeX derivation:

In [ ]:
rooti_ϕ = ϕ*sympy.sqrt(im)
# Check integral of surface pressure term for bottom stress:
tmp_τ_integrand = expand( (1 + exp(2*rooti_ϕ*ξ)) * exp(-rooti_ϕ*(ξ-H(x,y))) / (1 + exp(2*rooti_ϕ*H(x,y))) )
tmp = simplify(sympy.integrate(tmp_τ_integrand,(ξ,-H(x,y),0)).args[1].args[1])
Latex_tmp = (1/rooti_ϕ) * tanh(rooti_ϕ*H(x,y))
@assert tmp == Latex_tmp
println("LaTeX integral of the τb surface pressure term matches.")

# Check S(x) function:
Latex_S_fn = Latex_tmp
XXX = symbols("XXX")                            # SymPy can't pull the constant psg factor outside the integral.
τb_tmp = expand(τb.subs(psg(x,y),XXX).doit())
psg_coeff = τb_tmp.coeff(XXX)
testS = simplify(Latex_S_fn - simplify(psg_coeff))
@assert testS == 0
println("LaTeX S(x) function matches.")

# Check T(x) function:
function make_Latex_T_fn(int_bg_term)
    tmp_T_integrand2 = expand(tmp_τ_integrand * int_bg_term)
    tmp_T_fn = ( sympy.integrate(tmp_T_integrand2,(ξ,-H(x,y),0)) - τs(x,y)*(2*exp(rooti_ϕ*H(x,y)))/(1 + exp(2*rooti_ϕ*H(x,y))))
    return tmp_T_fn
end
int_bg_term = sympy.integrate(bg(x,y,ξ),(ξ,ξ,0))        # Generic integral of bg.
Latex_T_fn = make_Latex_T_fn(int_bg_term)

τb_tmp2 = (τb_tmp - psg_coeff * XXX).doit()
testT = τb_tmp2 - Latex_T_fn
YYY = symbols("YYY")                            # SymPy can't work on the buried bg integral, so substitute for it
testT = testT.subs(bg(x,y,ξ),YYY)
testT = expand(testT).doit()
@assert testT == 0
println("LaTeX T(x) function matches.")

# Check integral of surface pressure term for 𝔘:
tmp_ps_integrand = expand(exp(-rooti_ϕ*ξ)*(exp(rooti_ϕ*ξ) - exp(rooti_ϕ*H(x,y)))*(exp(rooti_ϕ*(ξ + H(x,y))) - 1))
tmp = sympy.integrate(tmp_ps_integrand,(ξ,-H(x,y),0))
Latex_tmp = - H(x,y) * (exp(2*rooti_ϕ*H(x,y)) + 1) + (1/rooti_ϕ)*(exp(2*rooti_ϕ*H(x,y)) - 1)
@assert tmp == Latex_tmp
println("LaTeX integral of the 𝔘 surface pressure term matches.")

# Check A(x) function:
Latex_A_fn = (im/f)*(H(x,y) + (1/rooti_ϕ)*((1 - exp(2*rooti_ϕ*H(x,y))) / (1 + exp(2*rooti_ϕ*H(x,y)) ) )).subs(ϕ,sqrt(f/ν₀)/ϵ)
𝔘_tmp = expand(𝔘.subs(psg(x,y),XXX).doit())
psg_coeff = 𝔘_tmp.coeff(XXX).subs(ϕ,sqrt(f/ν₀)/ϵ)
testA = simplify(Latex_A_fn - psg_coeff)
@assert testA == 0
println("LaTeX A(x) function matches.")

# Check B(x) function:
function make_Latex_B_fn(int_bg_term)
    tmp_B_integrand2 = expand(tmp_ps_integrand * int_bg_term)
    tmp_B_fn = (-im/(f*(1+exp(2*rooti_ϕ*H(x,y))))) * ( sympy.integrate(tmp_B_integrand2,(ξ,-H(x,y),0)) + τs(x,y)*(exp(rooti_ϕ*H(x,y)) - 1)^2)
    return tmp_B_fn
end
Latex_B_fn = make_Latex_B_fn(int_bg_term)
𝔘_tmp2 = (𝔘_tmp - psg_coeff * XXX).doit()
testB = (𝔘_tmp2 - Latex_B_fn).subs(ϕ,sqrt(f/ν₀)/ϵ)
testB = testB.subs(bg(x,y,ξ),YYY)
testB = expand(testB).doit()
@assert testB == 0
println("LaTeX B(x) function matches.")

### Define parameter values:

In [ ]:
# Define the parameter values
f_val  = 1
ϵ_val  = 0.95
ν₀_val = 0.06
κ₀_val = ν₀_val
γ_val  = 0.2
println()
println("Non-dimensional Ekman-layer depth:")
Ekman_depth = sqrt(2*ν₀_val/f_val)
display(Ekman_depth)

# Compute compound parameter:
ϕ_val = sqrt(f_val / ν₀_val) / ϵ_val
ψ_val = sqrt(γ_val / κ₀_val) / ϵ_val

# Define domain depth:
H_val(x,y)    = 1.0 - x^2 - y^2

# Define wind stress:
# τs_val(x,y) = 0.1 + 0.0im
function τs_val(x, y)
    r = sqrt(x^2 + y^2)
    V = 0.0 * r^2           # Define this function for your use-case
    return -V * y / r + im* V * x / r
end

# Define relaxation buoyancy field:
α, β   = symbols("α  β",  real=true, positive=true) # Relaxation buoyancy field parameters
B_val(x,y,ξ)  = α * ξ
α_val = 0.2
# B_val(x,y,ξ)  = α * (exp(β * ξ) - 1)            # Aspirational
B_valH(x, y)  = B_val(x,y,-H_val(x,y) )

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val, ϵ=>ϵ_val, ν₀=>ν₀_val, ϕ=>ϕ_val, γ=>γ_val, κ₀=>κ₀_val, ψ=>ψ_val, α=>α_val)
fn_values    = Dict(
    τs(x,y)=>τs_val(x,y), 
    H(x, y)=>H_val(x,y)
    )
B_values     = Dict(
    B(x, y, ξ)=>B_val(x, y, ξ), 
    B(x, y, z)=>B_val(x, y, z), 
    B(x, y, -H(x,y))=>B_valH(x,y)
    )

# Compute baroclinic pressure gradient term:
@time begin
    println()
    println("Computing baroclinic pressure gradient term...")
    this_b = simplify(b.subs(B_values).doit())
    this_pbarog = simplify(diff(this_b,x) + im*diff(this_b,y))
    this_pbarog_intz = simplify(sympy.integrate(expand(this_pbarog),(z,ξ,0)))         # This isn't too crazy!
    B_values[SymPy.integrate(bg(x,y,ξ),(ξ,ξ,0))] = this_pbarog_intz

    Latex_T_fn = make_Latex_T_fn(this_pbarog_intz)
    Latex_T_fn = Latex_T_fn.subs(param_values).subs(B_values).subs(fn_values).doit()
    Latex_B_fn = make_Latex_B_fn(this_pbarog_intz)
    Latex_B_fn = Latex_B_fn.subs(param_values).subs(B_values).subs(fn_values).doit()
    println("done.")
end

println()
println("This case parameter values:")
display(param_values)

println()
println("This case function values:")
display(fn_values)

println()
println("This relaxation buoyancy function values:")
display(B_values)

### Solve for surface pressure using Gridap

In [ ]:
println()
println("Solving for the surface pressure using Gridap...")

@time begin

# 1. Define the mesh
    model = GmshDiscreteModel("unit_circle_v0.3.msh")
    # model = GmshDiscreteModel("unit_circle_v0.2.msh")
    Ω = Triangulation(model)
    dΩ = Measure(Ω, 2)

# 2. Define the finite element space (piecewise linear,Dirichlet zero BC)
    order = 2
    reffe = ReferenceFE(lagrangian, Float64, order)

    V = TestFESpace(model, reffe; conformity=:H1, dirichlet_tags="boundary")
    U = TrialFESpace(V)

# 3. Define the coefficients of the elliptic equation:
    A_fn_tmp = lambdify(Latex_A_fn.subs(fn_values).subs(param_values).doit(), [x, y])
    A_fn(xx) = (typeof(xx[1]) <: Real ? A_fn_tmp(xx[1], xx[2]) : 1.0)     # Might get called with non-Float argument
    B_fn_tmp = lambdify(Latex_B_fn.subs(fn_values).subs(param_values).doit(), [x, y])   # This is where the barolinic pressure gradient term is inserted
    B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp(xx[1], xx[2]) : 0.0)     # Might get called with non-Float argument

# 4. Define weak form (variational formulation)
    a(u,v) = ∫( real( (∇(v) ⋅ VectorValue( 1.0, -1im)) * (A_fn * (∇(u) ⋅ VectorValue(1.0, 1im))) ) )dΩ
    l(v)   = ∫( real( (∇(v) ⋅ VectorValue(-1.0,  1im)) *  B_fn ) )dΩ

# 5. Assemble and solve
    op = AffineFEOperator(a, l, U, V)
    psurf = Gridap.solve(op)  # This is your numerical solution as a Gridap FEFunction
end

# 6. Visualization with Paraview
writevtk(Ω,notebook_name * "_psurf_solution",cellfields=["psurf"=>psurf])

#### Setup helper functions:

In [ ]:
function ps_val(xx, yy)
    try
    	gp = evaluate(psurf, Point(xx, yy))
	    return gp
    catch
        return 0.0
    end
end

function psg_val(xx, yy)
    try
	    tmp = evaluate(∇(psurf), Point(xx, yy))
	    return tmp[1] + 1im*tmp[2] 
    catch
        return 0.0 + 1im*0.0
    end
end

function UV_fn(xx,yy) 
    try
        psg_tmp = evaluate(∇(psurf), Point(xx, yy))
        UV = A_fn_tmp(xx,yy)*(psg_tmp[1] + 1im*psg_tmp[2]) + B_fn_tmp(xx,yy)
        return UV
    catch
        return 0.0 + 1im*0.0
    end
end

### Solve for velocity field using the Green's function:

In [ ]:
println()
println("Computing velocity field from surface pressure, Green's function, and known parameter and fields:")

# Make Julia functions from the symbolic expressions to accelerate for loop:
Guv0_fn             = lambdify(Guv0.subs(         param_values).subs(fn_values), [x, y, z])
Guv_int_wrt_ξ_fn    = lambdify(Guv_int_wrt_ξ.subs(param_values).subs(fn_values), [x, y, z])
S_fn_tmp		    = lambdify(Latex_S_fn.subs(   param_values).subs(fn_values).doit(), [x, y])
T_fn_tmp		    = lambdify(Latex_T_fn, [x, y])
τb_fn(xx,yy)        = S_fn_tmp(xx,yy)*psg_val(xx,yy) + T_fn_tmp(xx,yy)
b_fn(xx,yy,zz)      = lambdify(b.subs(               fn_values).subs(B_values).subs(ξ,z).subs(param_values).doit(), [x, y, z])(xx,yy,zz)
pbarog_fn(xx,yy,zz) = lambdify(this_pbarog_intz.subs(fn_values).subs(B_values).subs(ξ,z).subs(param_values).doit(), [x, y, z])(xx,yy,zz)

# Nx, Ny, Nz = 128, 128, 64
Nx, Ny, Nz = 64, 64, 32
# Nx, Ny, Nz = 32, 32, 16
xs     = range(-1, 1, Nx)
ys     = range(-1, 1, Ny)
zs     = range(-1, 0, Nz)
us     = zeros(Nx, Ny, Nz)
vs     = zeros(Nx, Ny, Nz)
bs     = zeros(Nx, Ny, Nz)
Us     = zeros(Nx, Ny)
Vs     = zeros(Nx, Ny)
τxs    = zeros(Nx, Ny)
τys    = zeros(Nx, Ny)
τbxs   = zeros(Nx, Ny)
τbys   = zeros(Nx, Ny)
pss    = zeros(Nx, Ny)
spds   = zeros(Nx, Ny)

@warn "14Dec25: Code to compute 3D velocity field is incorrect! 
The baroclinic pressure gradient term needs to be integrated over ξ from -H to 0, for every z.
This was done wrong for the Arctic Ocean Dynamics Workshop example solution.
See PlanetaryGoestrophySolutions_v0.6.tex and ExampleSolution_v0.5.ipynb for correct derivation.
The 2D fields are still correct."

@time begin
# @profile begin
# @profview begin
for (ix, xx) in enumerate(xs)
	for (iy, yy) in enumerate(ys)
		this_H = H_val(xx, yy)
		if this_H >= 0
			this_pss_val = ps_val( xx, yy)      # Interpolate or evaluate ps
			this_psg_val = psg_val(xx, yy)      # Interpolate or evaluate psg
			for (iz, zz) in enumerate(zs)
				if zz <= 0 && zz >= -this_H
					# Pressure driven part

					# WARNING! This part is incorrect. The baroclinic pressure gradient
					# term needs to be integrated over ξ from -H to 0, for every z
					this_pbarog_val = pbarog_fn(xx, yy, zz)
					Iuv = Guv_int_wrt_ξ_fn(xx, yy, zz) * (this_psg_val + this_pbarog_val)

					# Surface stress driven part
					S = Guv0_fn(xx, yy, zz) * τs_val(xx, yy)
					us[ix, iy, iz] = real(Iuv - S)
					vs[ix, iy, iz] = imag(Iuv - S)
					bs[ix, iy, iz] = b_fn(xx, yy, zz)
				end
			end
			# Compute 2D fields here:
			UV             = UV_fn(xx,yy)
			Us[    ix, iy] = float(real(UV))
			Vs[    ix, iy] = float(imag(UV))
			spds[  ix, iy] = sqrt(Us[ix, iy]^2 + Vs[ix, iy]^2)
			τxs[   ix, iy] = float(real(τs_val(xx,yy)))
			τys[   ix, iy] = float(imag(τs_val(xx,yy)))
			τb_val         = τb_fn(xx,yy)
			τbxs[  ix, iy] = float(real(τb_val))
			τbys[  ix, iy] = float(imag(τb_val))
			pss[   ix, iy] = this_pss_val
		end
	end
end
end

# Write out solution for display by Paraview:
vtk_grid(notebook_name * "_3D_solution", xs, ys, zs) do vtk
	vtk["u_speed"]  = us
	vtk["v_speed"]  = vs
	vtk["b_field"]  = bs
end

vtk_grid(notebook_name * "_2D_solution", xs, ys) do vtk
	vtk["U_speed"]      = Us
	vtk["V_speed"]      = Vs
	vtk["speed"]        = spds
	vtk["x_sfc_stress"] = τxs
	vtk["y_sfc_stress"] = τys
	vtk["x_bot_stress"] = τbxs
	vtk["y_bot_stress"] = τbys
	vtk["sfc_p"]        = pss
end